In [1]:
import torch
import pandas as pd
import numpy as np
from sklearn.preprocessing import normalize
from sentence_transformers import SentenceTransformer
import re
import unicodedata
from pymilvus import FieldSchema, CollectionSchema, DataType, Collection, utility, connections

In [3]:
df = pd.read_csv('../data/benchmark/queries.csv')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46 entries, 0 to 45
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   query_id    46 non-null     int64 
 1   query_text  46 non-null     object
dtypes: int64(1), object(1)
memory usage: 868.0+ bytes


In [4]:
def clean_vietnamese_text(text):
    if not isinstance(text, str):
        return ""
        
    text = unicodedata.normalize('NFC', text)
    text = text.lower()
    text = text.replace('%', ' phần trăm')
    
    text = re.sub(r'/\s*đêm', ' mỗi đêm', text, flags=re.IGNORECASE)
    text = re.sub(r"[^\w\s\-,.]", " ", text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [5]:
df["QueryForEmbedding"] = df["query_text"].apply(clean_vietnamese_text)

In [6]:
# model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-mpnet-base-v2')
# model = SentenceTransformer('embaas/sentence-transformers-multilingual-e5-base')
model = SentenceTransformer('embaas/sentence-transformers-multilingual-e5-large')
# model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

In [7]:
embeddings = model.encode(df["QueryForEmbedding"].tolist(), batch_size=32, show_progress_bar=True)
embeddings = normalize(embeddings, axis=1)  # chuẩn hóa theo hàng

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

In [8]:
print("Embedding shape:", embeddings.shape)

Embedding shape: (46, 1024)


In [9]:
norms = np.linalg.norm(embeddings, axis=1)
print(norms[:10])

[1.         1.0000001  1.         0.9999999  1.0000001  1.
 0.99999994 1.0000001  1.0000002  1.0000001 ]


In [11]:
connections.connect(alias="default",host="localhost",port="19530")
collection_name = "queries_e5_large_v2"

In [12]:
if utility.has_collection(collection_name):
    print(f"Collection '{collection_name}' đã tồn tại, xóa và tạo mới lại...")
    utility.drop_collection(collection_name)

# Tạo mới collection
print(f"Tạo mới collection '{collection_name}'...")

fields = [
    FieldSchema(name="query_id", dtype=DataType.INT64, is_primary=True, auto_id=False),
    FieldSchema(name="QueryForEmbedding", dtype=DataType.FLOAT_VECTOR, dim=1024),
    FieldSchema(name="query_text", dtype=DataType.VARCHAR, max_length=512),
]

schema = CollectionSchema(fields=fields, description="Query dataset embeddings")

collection = Collection(name=collection_name, schema=schema)
print(f"Collection '{collection_name}' đã được tạo mới thành công!")

Collection 'queries_e5_large_v2' đã tồn tại, xóa và tạo mới lại...
Tạo mới collection 'queries_e5_large_v2'...
Collection 'queries_e5_large_v2' đã được tạo mới thành công!


In [13]:
query_ids = df["query_id"].astype(int).tolist()
query_texts = df["query_text"].str.strip().tolist()

In [14]:
print(type(embeddings))
print(type(query_ids))
print(type(query_texts))

<class 'numpy.ndarray'>
<class 'list'>
<class 'list'>


In [15]:
def chunk_data(data, size):
    for i in range(0, len(data), size):
        yield data[i:i+size]

batch_size = 100

for i, (id_batch, emb_batch, q_batch) in enumerate(zip(
    chunk_data(query_ids, batch_size),
    chunk_data(embeddings.tolist(), batch_size),
    chunk_data(query_texts, batch_size)
)):
    collection.insert([
        id_batch,
        emb_batch,
        q_batch
    ])
    print(f"Inserted batch {i+1}")

collection.flush()
print(f"Số lượng bản ghi: {collection.num_entities}")

Inserted batch 1
Số lượng bản ghi: 46


In [16]:
index_params = {
    "index_type": "HNSW",
    "metric_type": "COSINE",
    "params": {"M": 8, "efConstruction": 64}
}
# Tạo index cho field vector
collection.release()
collection.drop_index()
collection.create_index(field_name="QueryForEmbedding", index_params=index_params)
collection.load()